# 1. Load Models, Tokenizer, and Evaluation Dataset

Load the base model, tokenizer, required paths, and the 100-question finance evaluation dataset used throughout the evaluation pipeline.

In [ ]:
import os
import json
import re
import time
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
 
PROJECT_ROOT = Path('/kaggle/working/FinAlign')
BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
SFT_ADAPTER = PROJECT_ROOT / 'checkpoints' / 'sft_adapter'
DPO_ADAPTER = Path('/kaggle/working/dpo_adapter')
EVAL_FILE = PROJECT_ROOT / 'data' / 'eval_set' / 'custom_finance_eval.jsonl'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.environ.get('HF_TOKEN')

print('CUDA:', torch.cuda.is_available())
print('Evaluation questions file:', EVAL_FILE.exists())

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16,
    device_map='auto',
    token=HF_TOKEN
)

print('Base model loaded')


def load_adapter_model(base, adapter_path):
    m = PeftModel.from_pretrained(base, str(adapter_path), is_trainable=False)
    m.eval()
    return m


def generate_response(model, question, max_new_tokens=150):
    inputs = tokenizer(question, return_tensors='pt', truncation=True).to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(
        output[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

with open(EVAL_FILE, encoding='utf-8') as f:
    finance_eval = [json.loads(line) for line in f]

print('Evaluation questions:', len(finance_eval))



# 2. Load the SFT Adapter

Load the trained SFT LoRA adapter on top of the base model and prepare it for inference.

In [18]:
import os
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
SFT_ADAPTER = "/kaggle/working/FinAlign/checkpoints/sft_adapter"

HF_TOKEN = os.environ.get("HF_TOKEN")

print("Base model:", BASE_MODEL)
print("SFT adapter exists:", os.path.exists(SFT_ADAPTER))

# Load tokenizer
print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    token=HF_TOKEN
)

# Load base Qwen2.5-3B
print("Loading Qwen 2.5 3B...")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16,
    device_map="auto",
    token=HF_TOKEN
)

# Attach SFT LoRA adapter
print("Loading SFT adapter...")

sft_model = PeftModel.from_pretrained(
    base_model,
    SFT_ADAPTER
)

sft_model.eval()

print("\n✅ SFT MODEL LOADED SUCCESSFULLY!")

Base model: mistralai/Mistral-7B-Instruct-v0.2
SFT adapter exists: True

Loading tokenizer...
Loading Mistral 7B...


Loading weights:   0%|          | 0/291 [00:00<, it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<, B/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['monteclora_config', 'velora_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading SFT adapter...

✅ SFT MODEL LOADED SUCCESSFULLY!


# 3. Define Response Quality Checks

Define the response generation and quality-checking utilities used to detect issues such as empty or potentially truncated responses.

In [ ]:
import re

def check_response(response):
    response_lower = response.lower()

    issues = []

    # Empty/very short answer
    if len(response.strip()) < 50:
        issues.append("too_short")

    # Possible truncation
    if response.rstrip().endswith((
        "and",
        "or",
        "the",
        "to",
        "of",
        "in",
        "for",
        "with",
        "is",
        "are",
        "can",
        "may"
    )):
        issues.append("possibly_truncated")

    # Prompt/template leakage
    bad_patterns = [
        "[inst]",
        "[/inst]",
        "<|assistant|>",
        "<|user|>",
        "[yourname]",
        "yourname"
    ]

    for pattern in bad_patterns:
        if pattern in response_lower:
            issues.append("template_leakage")
            break

    # Excessive repetition
    sentences = re.split(r'[.!]\s+', response)
    sentences = [s.strip().lower() for s in sentences if s.strip()]

    if len(sentences) >= 6:
        unique_sentences = len(set(sentences))
        repetition_ratio = unique_sentences / len(sentences)

        if repetition_ratio < 0.7:
            issues.append("repetitive")

    return issues

print("Response quality checker defined.")



DPO AUTOMATIC QUALITY CHECK
Total responses: 100
Responses with possible issues: 5
Responses without detected issues: 95


# 4. Generate SFT and DPO Responses

Generate responses from the SFT and DPO models for a fixed set of evaluation questions to compare their output quality.

In [ ]:
# Compare SFT vs DPO on 5 evaluation questions

import json
import time

EVAL_FILE = PROJECT_ROOT / "data" / "eval_set" / "custom_finance_eval.jsonl"

# Load first 5 evaluation questions
test_questions = []

with open(EVAL_FILE, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5:
            break

        item = json.loads(line)

        test_questions.append({
            "id": item.get("id", i + 1),
            "question": item["question"]
        })

print("=" * 70)
print("STEP 15: SFT vs DPO COMPARISON")
print("=" * 70)
print(f"Questions: {len(test_questions)}")
print()


# ---------------------------------------------------------
# Load DPO adapter & generate responses
# ---------------------------------------------------------

print("Loading DPO adapter...")
dpo_model = load_adapter_model(base_model, DPO_ADAPTER)
print("✓ DPO model loaded")

print("\nGenerating DPO responses...")
dpo_results = []

for item in test_questions:

    start = time.time()

    response = generate_response(
        dpo_model,
        item["question"],
        max_new_tokens=150
    )

    elapsed = time.time() - start

    dpo_results.append({
        "id": item["id"],
        "question": item["question"],
        "response": response
    })

    print(f"✓ DPO Question {item['id']} completed in {elapsed:.1f}s")


# ---------------------------------------------------------
# Load SFT adapter & generate responses
# ---------------------------------------------------------

print("\nLoading SFT adapter...")

SFT_ADAPTER_PATH = PROJECT_ROOT / "checkpoints" / "sft_adapter"

sft_model = load_adapter_model(
    base_model,
    SFT_ADAPTER_PATH
)

print("✓ SFT model loaded")


print("\nGenerating SFT responses...")

sft_results = []

for item in test_questions:

    start = time.time()

    response = generate_response(
        sft_model,
        item["question"],
        max_new_tokens=150
    )

    elapsed = time.time() - start

    sft_results.append({
        "id": item["id"],
        "question": item["question"],
        "response": response
    })

    print(f"✓ SFT Question {item['id']} completed in {elapsed:.1f}s")


print("\n" + "=" * 70)
print("SFT vs DPO TEST COMPLETED")
print("=" * 70)



STEP 15: SFT vs DPO COMPARISON
Questions: 5

Generating DPO responses...
✓ DPO Question 1 completed in 12.8s
✓ DPO Question 2 completed in 10.4s
✓ DPO Question 3 completed in 12.7s
✓ DPO Question 4 completed in 11.9s
✓ DPO Question 5 completed in 12.9s

Loading SFT adapter...
Loading adapter from: /kaggle/working/FinAlign/checkpoints/sft_adapter


/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['monteclora_config', 'velora_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adapter loaded successfully!
✓ SFT model loaded

Generating SFT responses...
✓ SFT Question 1 completed in 6.0s
✓ SFT Question 2 completed in 12.7s
✓ SFT Question 3 completed in 11.7s
✓ SFT Question 4 completed in 12.6s
✓ SFT Question 5 completed in 12.6s

SFT vs DPO TEST COMPLETED


# 5. Compare SFT and DPO Response Quality

Compare SFT and DPO responses using response length and detected quality issues, and summarize the evaluation results.

In [ ]:
# SFT vs DPO QUALITY COMPARISON

import re
import json
from pathlib import Path

print("=" * 70)
print("STEP 16: SFT vs DPO QUALITY COMPARISON")
print("=" * 70)

dpo_responses = dpo_results
sft_responses = sft_results

print(f"\nUsing DPO responses: {len(dpo_responses)} items")
print(f"Using SFT responses: {len(sft_responses)} items")


# Convert responses to plain text
def extract_text(response):

    if isinstance(response, str):
        return response

    if isinstance(response, dict):

        for key in ["response", "text", "generated_text", "answer"]:
            if key in response:
                return str(response[key])

        return json.dumps(response, ensure_ascii=False)

    return str(response)


dpo_text = [extract_text(x) for x in dpo_responses]
sft_text = [extract_text(x) for x in sft_responses]


# Basic quality checks
def quality_metrics(text):

    words = len(text.split())
    chars = len(text)

    issues = []

    # Very short answer
    if words < 40:
        issues.append("too_short")

    # Possible generation truncation
    if text.rstrip().endswith(("...", "…", ":")):
        issues.append("possibly_truncated")

    # Instruction/prompt leakage
    leakage_patterns = [
        "[INST]",
        "[/INST]",
        "<s>",
        "</s>",
        "### Instruction",
        "### Response"
    ]

    for pattern in leakage_patterns:
        if pattern.lower() in text.lower():
            issues.append("prompt_leakage")
            break

    # Empty response
    if words == 0:
        issues.append("empty")

    return {
        "words": words,
        "characters": chars,
        "issues": issues
    }


# Calculate metrics
dpo_metrics = [quality_metrics(x) for x in dpo_text]
sft_metrics = [quality_metrics(x) for x in sft_text]


# Print question-by-question comparison
print("\n")
print("=" * 70)
print("QUESTION-BY-QUESTION COMPARISON")
print("=" * 70)

for i in range(min(len(dpo_text), len(sft_text))):

    print(f"\nQuestion {i + 1}")
    print("-" * 70)

    print(f"SFT: {sft_metrics[i]['words']} words")
    print(f"DPO: {dpo_metrics[i]['words']} words")
    print(f"SFT issues: {sft_metrics[i]['issues'] if sft_metrics[i]['issues'] else 'None'}")
    print(f"DPO issues: {dpo_metrics[i]['issues'] if dpo_metrics[i]['issues'] else 'None'}")


# Overall statistics
sft_avg_words = sum(x["words"] for x in sft_metrics) / len(sft_metrics)
dpo_avg_words = sum(x["words"] for x in dpo_metrics) / len(dpo_metrics)

sft_issues = sum(len(x["issues"]) > 0 for x in sft_metrics)
dpo_issues = sum(len(x["issues"]) > 0 for x in dpo_metrics)


print("\n")
print("=" * 70)
print("OVERALL COMPARISON")
print("=" * 70)

print(f"Questions compared: {min(len(sft_text), len(dpo_text))}")

print("\nSFT:")
print(f"  Average response length : {sft_avg_words:.1f} words")
print(f"  Responses with issues   : {sft_issues}")

print("\nDPO:")
print(f"  Average response length : {dpo_avg_words:.1f} words")
print(f"  Responses with issues   : {dpo_issues}")


# Determine simple winner
print("\n")
print("=" * 70)
print("RESULT")
print("=" * 70)

if dpo_issues < sft_issues:
    print("DPO shows fewer detected quality issues than SFT.")
elif sft_issues < dpo_issues:
    print("SFT shows fewer detected quality issues than DPO.")
else:
    print("SFT and DPO have the same number of detected issues.")



STEP 16: SFT vs DPO QUALITY COMPARISON

DPO candidate variables:
  dpo_results: 5 items

SFT candidate variables:
  sft_results: 5 items

Using DPO variable: dpo_results

Using SFT variable: sft_results


QUESTION-BY-QUESTION COMPARISON

Question 1
----------------------------------------------------------------------
SFT: 58 words
DPO: 114 words
SFT issues: None
DPO issues: None

Question 2
----------------------------------------------------------------------
SFT: 112 words
DPO: 87 words
SFT issues: None
DPO issues: None

Question 3
----------------------------------------------------------------------
SFT: 101 words
DPO: 93 words
SFT issues: None
DPO issues: None

Question 4
----------------------------------------------------------------------
SFT: 73 words
DPO: 85 words
SFT issues: None
DPO issues: None

Question 5
----------------------------------------------------------------------
SFT: 115 words
DPO: 110 words
SFT issues: None
DPO issues: None


OVERALL COMPARISON
Questions co

# 6. Evaluate the Merged DPO Model

Evaluate the final merged DPO model on the complete 100-question finance evaluation dataset and save the generated responses.

In [ ]:
# Load and merge DPO adapter for full evaluation
print("Loading base model and merging DPO adapter...")
base_model_for_merge = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16,
    device_map="auto",
    token=HF_TOKEN
)
peft_dpo = PeftModel.from_pretrained(base_model_for_merge, DPO_ADAPTER)
model = peft_dpo.merge_and_unload()
model.eval()

print("✓ Merged DPO model ready for evaluation")


In [63]:
import json

results = []

for i, item in enumerate(finance_eval):
    question = item["question"]

    inputs = tokenizer(
        question,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False
        )

    response = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    results.append({
        "id": item.get("id", i + 1),
        "question": question,
        "response": response
    })

    if (i + 1) % 10 == 0:
        print(f"Evaluated {i + 1}/100")



Evaluated 10/100
Evaluated 20/100
Evaluated 30/100
Evaluated 40/100
Evaluated 50/100
Evaluated 60/100
Evaluated 70/100
Evaluated 80/100
Evaluated 90/100
Evaluated 100/100


In [ ]:
# Run automatic quality checks on evaluated merged DPO responses
total = len(results)
problem_count = 0

for result in results:
    issues = check_response(result["response"])
    result["automatic_issues"] = issues

    if issues:
        problem_count += 1

print("=" * 60)
print("DPO AUTOMATIC QUALITY CHECK")
print("=" * 60)
print("Total responses:", total)
print("Responses with possible issues:", problem_count)
print("Responses without detected issues:", total - problem_count)

# Calculate evaluation statistics
issue_count = sum(1 for r in results if r.get("automatic_issues"))
clean_count = total - issue_count
issue_rate = (issue_count / total) * 100
clean_rate = (clean_count / total) * 100

print("=" * 70)
print("DPO EVALUATION STATISTICS")
print("=" * 70)
print(f"Total responses:              {total}")
print(f"Responses without issues:     {clean_count}")
print(f"Responses with issues:        {issue_count}")
print(f"Clean response rate:          {clean_rate:.2f}%")
print(f"Possible issue rate:           {issue_rate:.2f}%")
print("=" * 70)


# 7. Generate Final Evaluation Report

Calculate the final evaluation statistics and save the comparison results and final evaluation report for the project.

In [ ]:
import json
from pathlib import Path

RESULT_FILE = Path("/kaggle/working/FinAlign/reports/dpo_merged_evaluation.jsonl")

with open(RESULT_FILE, "w", encoding="utf-8") as f:
    for result in results:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")

print("Saved:", RESULT_FILE)
print("Results:", len(results))


responses = []

with open("/kaggle/working/FinAlign/reports/dpo_merged_evaluation.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        responses.append(json.loads(line))

word_counts = [
    len(item["response"].split())
    for item in responses
]

print("Responses:", len(responses))
print("Average words:", round(sum(word_counts) / len(word_counts), 1))
print("Minimum words:", min(word_counts))
print("Maximum words:", max(word_counts))


report_dir = Path("/kaggle/working/FinAlign/reports")
report_dir.mkdir(exist_ok=True)

comparison = {
    "sft": {
        "responses": len(sft_results),
        "average_words": sft_avg_words,
        "issues": sft_issues
    },
    "dpo": {
        "responses": len(dpo_results),
        "average_words": dpo_avg_words,
        "issues": dpo_issues
    },
    "dpo_merged": {
        "responses": len(results),
        "average_words": 103.1
    }
}

with open(report_dir / "final_comparison.json", "w") as f:
    json.dump(comparison, f, indent=2)

print("Step 19 completed")

print(comparison)


report_dir = Path("/kaggle/working/FinAlign/reports")

final_report = {
    "sft": {
        "test_questions": len(sft_results),
        "average_words": sft_avg_words,
        "issues": sft_issues
    },
    "dpo": {
        "test_questions": len(dpo_results),
        "average_words": dpo_avg_words,
        "issues": dpo_issues
    },
    "dpo_merged": {
        "evaluation_questions": len(results),
        "average_words": 103.1,
        "evaluation_file": str(report_dir / "dpo_merged_evaluation.jsonl")
    }
}

with open(report_dir / "final_evaluation_report.json", "w") as f:
    json.dump(final_report, f, indent=2)

print("Step 20 completed")
print("Saved:", report_dir / "final_evaluation_report.json")

Saved: /kaggle/working/FinAlign/reports/dpo_merged_evaluation.jsonl
Results: 100
